# Complete two-threshold cross-threshold event relationships

This notebook maps every completed small-threshold event to the causal large-threshold context available at its DCC (USDJPY at 0.25% / 0.50%; see the configuration cell), compares agreement and disagreement, reconstructs small-event sequences inside large-mode intervals, and asks whether initiating small-event geometry differs when a counter-scale attempt is later propagated or rejected.

The grouping label is a future outcome and therefore diagnostic hindsight. No HMM, predictive model, or trading rule is fitted here.

This lab deliberately uses a different market and threshold pair from `2_threshold_dc.ipynb` (EURUSD at 0.20% / 0.40%), so a difference between the two notebooks mixes a symbol change with a threshold change. `2_threshold_dc_robustness.ipynb` runs both symbols at several predeclared pairs on one common window to separate those two changes.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dual_dc
from dc_basics.plotting import (
    plot_dc_attempt_feature_distributions,
    plot_dc_attempt_tmv_t_space,
    plot_dc_counter_scale_price_casebook,
    plot_dc_counter_scale_trajectories,
    plot_dc_relationship_feature_distributions,
    plot_two_threshold_directional_change,
)

In [2]:
from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


from dc_basics.data import resolve_data_root

DATA_ROOT = resolve_data_root()
SYMBOL = "USDJPY.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.0025
LARGE_THETA = 0.0050
N_PRICES = 5_000
FEATURE_COLUMNS = ["tmv", "t", "r"]

if not 0.0 < SMALL_THETA < LARGE_THETA:
    raise ValueError("Thresholds must satisfy 0 < SMALL_THETA < LARGE_THETA.")

print(f"Symbol: {SYMBOL}")
print(f"Small threshold: {SMALL_THETA:.3%}")
print(f"Large threshold: {LARGE_THETA:.3%}")

Symbol: USDJPY.r
Small threshold: 0.250%
Large threshold: 0.500%


## Semantic guardrail: which event do these features describe?

At a small downward DCC, the newly known completed event is the **up leg that just ended**. Its TMV, $T$, and $R$ do not measure the new downward attempt. Symmetrically, an upward DCC reveals the completed down leg.

Therefore, this notebook asks:

> Does the geometry of the small leg that just ended accompany a counter-scale reversal that is later propagated or rejected?

It does **not** yet ask how fast the new counter-scale move itself is developing. That requires live post-DCC measurements and is a separate later audit.

## Load and validate the price path

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,667
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


## Reconstruct the two event clocks and causal modes

This deliberately repeats the minimum mechanics from `2_threshold_dc.ipynb` so this notebook is independently executable.

In [4]:
from dc_basics.events import prepare_event_ledger


small_raw_events, large_raw_events = parse_dual_dc(prices, stheta=SMALL_THETA, btheta=LARGE_THETA)
small_events = prepare_event_ledger(small_raw_events, "S", "small", SMALL_THETA)
large_events = prepare_event_ledger(large_raw_events, "L", "large", LARGE_THETA)
small_live = dc_live_features(prices, theta=SMALL_THETA)
large_live = dc_live_features(prices, theta=LARGE_THETA)
mode_map = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

mode_frame = pd.DataFrame(index=prices.index)
mode_frame["small_mode"] = small_live["direction"].map(mode_map)
mode_frame["large_mode"] = large_live["direction"].map(mode_map)
for scale, live in (("small", small_live), ("large", large_live)):
    mode_frame[f"{scale}_upward_dcc"] = live["upward_dcc"]
    mode_frame[f"{scale}_downward_dcc"] = live["downward_dcc"]
mode_frame["small_event_id"] = mode_frame.index.to_series().map(
    small_events.set_index("dcc_idx")["event_id"]
)

print(f"Small completed events: {len(small_events):,}")
print(f"Large completed events: {len(large_events):,}")

Small completed events: 2,631
Large completed events: 840


## Master cross-threshold relationship table: every small DCC

Each row below is a completed small event indexed by its DCC information time. The table records two causal snapshots of the large clock:

- `previous_bar`: the large state immediately before processing the small-DCC bar;
- `at_dcc_close`: the large state after processing that bar, when both thresholds' same-bar confirmations are known.

On a simultaneous small/large DCC, the at-close large live features have reset into the new large mode. The previous-bar columns preserve the state that existed immediately before confirmation.

In [5]:
def build_small_dcc_relationships(events: pd.DataFrame, small_context: pd.DataFrame,
                                  large_context: pd.DataFrame) -> pd.DataFrame:
    relationships = events.copy()
    dcc_positions = relationships["dcc_pos"].to_numpy(dtype=np.int64)
    previous_large = large_context.shift(1)
    relationships["new_small_mode"] = relationships["direction"].map({"up": "down", "down": "up"})
    relationships["small_mode_at_dcc_close"] = (
        small_context["direction"].iloc[dcc_positions].map(mode_map).to_numpy()
    )
    relationships["large_mode_previous_bar"] = (
        previous_large["direction"].iloc[dcc_positions].map(mode_map).to_numpy()
    )
    relationships["large_mode_at_dcc_close"] = (
        large_context["direction"].iloc[dcc_positions].map(mode_map).to_numpy()
    )

    for feature in ("tmv_current", "t_current", "rdc_current"):
        short_name = feature.replace("_current", "")
        relationships[f"large_{short_name}_previous_bar"] = (
            previous_large[feature].iloc[dcc_positions].to_numpy(dtype=np.float64)
        )
        relationships[f"large_{short_name}_at_dcc_close"] = (
            large_context[feature].iloc[dcc_positions].to_numpy(dtype=np.float64)
        )

    relationships["large_dcc_same_bar"] = (
        large_context[["upward_dcc", "downward_dcc"]].any(axis=1).iloc[dcc_positions].to_numpy()
    )
    pre_context_known = relationships["large_mode_previous_bar"].notna()
    close_context_known = relationships["large_mode_at_dcc_close"].notna()
    relationships["completed_leg_relationship_before_dcc"] = pd.NA
    relationships.loc[pre_context_known, "completed_leg_relationship_before_dcc"] = np.where(
        relationships.loc[pre_context_known, "direction"].eq(
            relationships.loc[pre_context_known, "large_mode_previous_bar"]
        ),
        "agreement",
        "disagreement",
    )
    relationships["new_small_relationship_at_close"] = pd.NA
    relationships.loc[close_context_known, "new_small_relationship_at_close"] = np.where(
        relationships.loc[close_context_known, "new_small_mode"].eq(
            relationships.loc[close_context_known, "large_mode_at_dcc_close"]
        ),
        "agreement",
        "disagreement",
    )

    if not relationships["new_small_mode"].equals(relationships["small_mode_at_dcc_close"]):
        raise AssertionError("The event-implied new small mode disagrees with the causal live parser.")
    if not relationships["information_time"].eq(relationships["dcc_idx"]).all():
        raise AssertionError("A small-event relationship row is timestamped before its DCC.")
    return relationships


small_dcc_relationships = build_small_dcc_relationships(small_events, small_live, large_live)
relationship_transition_summary = small_dcc_relationships.groupby(
    ["completed_leg_relationship_before_dcc", "new_small_relationship_at_close"],
    dropna=False,
    sort=False,
).agg(
    small_events=("event_id", "size"),
    same_bar_large_dccs=("large_dcc_same_bar", "sum"),
)
relationship_transition_summary

,,small_events,same_bar_large_dccs
completed_leg_relationship_before_dcc,new_small_relationship_at_close,,
NaN,NaN,2,0
agreement,disagreement,1655,0
disagreement,agreement,894,0
agreement,agreement,80,80


In [6]:
relationship_columns = [
    "event_id", "information_time", "direction", "new_small_mode", "tmv", "t", "r",
    "large_mode_previous_bar", "large_tmv_previous_bar", "large_t_previous_bar",
    "large_rdc_previous_bar", "large_mode_at_dcc_close", "large_tmv_at_dcc_close",
    "large_t_at_dcc_close", "large_rdc_at_dcc_close", "large_dcc_same_bar",
    "completed_leg_relationship_before_dcc", "new_small_relationship_at_close",
]
small_dcc_relationships[relationship_columns].tail(10)

,event_id,information_time,direction,new_small_mode,tmv,t,r,large_mode_previous_bar,large_tmv_previous_bar,large_t_previous_bar,large_rdc_previous_bar,large_mode_at_dcc_close,large_tmv_at_dcc_close,large_t_at_dcc_close,large_rdc_at_dcc_close,large_dcc_same_bar,completed_leg_relationship_before_dcc,new_small_relationship_at_close
2621,S2622,2026-10-30 08:20:00+02:00,down,up,2.385847,179,0.000033,down,1.005533,431.0,0.000012,down,0.874157,432.0,0.000010,False,agreement,disagreement
2622,S2623,2026-10-30 11:15:00+02:00,up,down,1.338797,29,0.000115,down,1.136909,466.0,0.000012,down,1.312498,467.0,0.000014,False,disagreement,agreement
2623,S2624,2026-10-30 18:50:00+02:00,down,up,1.773189,76,0.000058,down,1.292287,557.0,0.000012,down,1.111644,558.0,0.000010,False,agreement,disagreement
2624,S2625,2026-11-04 02:20:00+02:00,up,down,4.484562,661,0.000017,up,1.777263,712.0,0.000012,up,1.686807,713.0,0.000012,False,agreement,disagreement
2625,S2626,2026-11-04 07:00:00+02:00,down,up,3.220299,103,0.000078,down,1.159106,107.0,0.000054,down,1.021777,108.0,0.000047,False,agreement,disagreement
2626,S2627,2026-11-04 19:30:00+02:00,up,down,2.992411,130,0.000058,up,1.014829,154.0,0.000033,up,0.953863,155.0,0.000031,False,agreement,disagreement
2627,S2628,2026-11-05 10:35:00+02:00,down,up,2.059971,155,0.000033,down,0.552183,205.0,0.000013,down,0.496713,206.0,0.000012,False,agreement,disagreement
2628,S2629,2026-11-05 17:00:00+02:00,up,down,1.672728,107,0.000039,down,0.674470,282.0,0.000012,down,0.708509,283.0,0.000013,False,disagreement,agreement
2629,S2630,2026-11-06 02:25:00+02:00,down,up,3.018563,121,0.000062,down,1.235478,395.0,0.000016,down,1.139666,396.0,0.000014,False,agreement,disagreement
2630,S2631,2026-11-09 09:05:00+02:00,up,down,3.135609,270,0.000029,up,1.108780,380.0,0.000015,up,1.036302,381.0,0.000014,False,agreement,disagreement


## Cross-threshold Audit 1: completed small-event geometry by pre-DCC relationship

This comparison classifies the completed small leg against the large mode on the previous bar. It is a causal snapshot available at the small DCC; it does not claim that the large mode remained unchanged for every bar of the completed small leg.

In [7]:
relationship_feature_long = small_dcc_relationships.dropna(
    subset=["completed_leg_relationship_before_dcc"]
).melt(
    id_vars=["completed_leg_relationship_before_dcc"],
    value_vars=FEATURE_COLUMNS,
    var_name="feature",
    value_name="value",
)
relationship_feature_summary = relationship_feature_long.groupby(
    ["completed_leg_relationship_before_dcc", "feature"], sort=False
)["value"].agg(
    count="size",
    mean="mean",
    p25=lambda values: values.quantile(0.25),
    median="median",
    p75=lambda values: values.quantile(0.75),
)
relationship_feature_summary.style.format(precision=6)

,,count,mean,p25,median,p75
completed_leg_relationship_before_dcc,feature,,,,,
agreement,tmv,1735,2.831315,1.811952,2.509451,3.425308
disagreement,tmv,894,1.418173,1.168524,1.374584,1.642954
agreement,t,1735,120.583285,28.000000,72.000000,154.000000
disagreement,t,894,55.092841,15.000000,33.000000,71.750000
agreement,r,1735,0.000269,0.000045,0.000086,0.000196
disagreement,r,894,0.000214,0.000049,0.000104,0.000226


In [8]:
small_relationship_fig = plot_dc_relationship_feature_distributions(
    relationships=small_dcc_relationships,
    feature_labels={"tmv": "log10(small TMV)", "t": "log10(small T)", "r": "log10(small R)"},
    relationship_column="completed_leg_relationship_before_dcc",
    title="Completed small-event geometry versus the previous-bar large mode",
)
small_relationship_fig.show()

## Cross-threshold Audit 2: live large-scale context at every small DCC

The large values below are live, unfinished measurements known at the small-DCC close. `large_rdc_at_dcc_close` is not completed-event R. Simultaneous large DCCs reset these live values, so their rows are counted separately and excluded from the distribution plot to avoid mixing pre- and post-reset semantics.

In [9]:
large_live_columns = [
    "large_tmv_at_dcc_close", "large_t_at_dcc_close", "large_rdc_at_dcc_close",
]
large_context_rows = small_dcc_relationships.dropna(
    subset=["new_small_relationship_at_close", *large_live_columns]
).copy()
large_context_summary = large_context_rows.groupby(
    ["new_small_relationship_at_close", "large_dcc_same_bar"], sort=False
).agg(
    small_dccs=("event_id", "size"),
    large_tmv_median=("large_tmv_at_dcc_close", "median"),
    large_t_median=("large_t_at_dcc_close", "median"),
    large_rdc_median=("large_rdc_at_dcc_close", "median"),
)
large_context_summary.style.format({
    "small_dccs": "{:,}",
    "large_tmv_median": "{:.4f}",
    "large_t_median": "{:.1f}",
    "large_rdc_median": "{:.3e}",
})

In [10]:
large_context_plot_rows = large_context_rows.loc[~large_context_rows["large_dcc_same_bar"]].copy()
positive_large_context = large_context_plot_rows[large_live_columns].gt(0.0).all(axis=1)
large_context_plot_rows = large_context_plot_rows.loc[positive_large_context]
large_relationship_fig = plot_dc_relationship_feature_distributions(
    relationships=large_context_plot_rows,
    feature_labels={
        "large_tmv_at_dcc_close": "log10(live large TMV)",
        "large_t_at_dcc_close": "log10(live large T)",
        "large_rdc_at_dcc_close": "log10(live large RDC)",
    },
    relationship_column="new_small_relationship_at_close",
    title="Live large-scale context when the small DCC closes in agreement or disagreement",
)
large_relationship_fig.show()

## Identify counter-scale attempts and their later resolution

The state machine starts an attempt when a small DCC changes the small mode against the large mode known immediately beforehand. A future large DCC means propagation; a future small DCC back toward the unchanged large mode means rejection.

In [11]:
from dc_basics.two_scale import build_disagreement_episodes


episodes = build_disagreement_episodes(mode_frame, prices)
episode_audit = episodes.groupby("outcome", dropna=False, sort=False).agg(
    episodes=("small_event_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
)
episode_audit

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,episodes,median_resolution_bars
outcome,,
rejected,894,33.0
propagated,840,21.5
unresolved_at_sample_end,1,NaN


## Build the causal-feature / hindsight-label table

Same-bar resolutions are contemporaneously known rather than future outcomes, and unresolved/ambiguous cases lack a clean binary future label. They are audited above but excluded from this diagnostic comparison.

In [12]:
future_binary_mask = (
    episodes["outcome"].isin([AttemptOutcome.PROPAGATED.value, AttemptOutcome.REJECTED.value])
    & episodes["resolution_bars"].gt(0)
)
resolved_future = episodes.loc[future_binary_mask].copy()
event_features = small_events[[
    "event_id", "direction", "information_time", "ext_start_idx", "ext_end_idx",
    "dcc_idx", "tmv", "t", "r",
]].rename(columns={
    "event_id": "small_event_id",
    "direction": "completed_leg_direction",
    "information_time": "feature_information_time",
})
attempts = resolved_future.merge(event_features, on="small_event_id", how="left", validate="one_to_one")
attempts = attempts.rename(columns={"outcome": "future_outcome"})
attempts["outcome_information_time"] = attempts["resolution_time"]
attempts["label_is_hindsight_at_start"] = attempts["outcome_information_time"].gt(attempts["start_time"])

if attempts[FEATURE_COLUMNS].isna().any().any():
    raise AssertionError("A counter-scale attempt failed to match its completed small event.")
if not attempts["feature_information_time"].eq(attempts["start_time"]).all():
    raise AssertionError("Completed-event features were not timestamped at the initiating DCC.")
if not attempts["label_is_hindsight_at_start"].all():
    raise AssertionError("The analysis table contains a label already known at the initiating DCC.")
np.testing.assert_allclose(
    attempts["r"],
    attempts["tmv"] * SMALL_THETA / attempts["t"],
    rtol=1e-12,
    atol=1e-15,
)
expected_completed_direction = attempts["new_small_mode"].map({"up": "down", "down": "up"})
if not attempts["completed_leg_direction"].equals(expected_completed_direction):
    raise AssertionError("Completed-leg direction is not opposite the new live small mode.")
if not attempts["completed_leg_direction"].equals(attempts["prior_large_mode"]):
    raise AssertionError("The completed small leg was not aligned with the prior large mode.")

episode_labels = episodes[[
    "small_event_id", "outcome", "resolution_time", "resolution_position", "resolution_bars",
]].rename(columns={
    "outcome": "future_outcome",
    "resolution_time": "outcome_information_time",
})
small_dcc_relationships = small_dcc_relationships.merge(
    episode_labels, left_on="event_id", right_on="small_event_id", how="left", validate="one_to_one"
)
small_dcc_relationships["is_counter_scale_attempt"] = small_dcc_relationships["future_outcome"].notna()
small_dcc_relationships["outcome_is_hindsight_at_dcc"] = pd.Series(
    pd.NA, index=small_dcc_relationships.index, dtype="boolean"
)
attempt_rows = small_dcc_relationships["is_counter_scale_attempt"]
small_dcc_relationships.loc[attempt_rows, "outcome_is_hindsight_at_dcc"] = (
    small_dcc_relationships.loc[attempt_rows, "outcome_information_time"].gt(
        small_dcc_relationships.loc[attempt_rows, "information_time"]
    )
)

print(f"Clean future-labelled attempts: {len(attempts):,}")
attempts[[
    "small_event_id", "start_time", "completed_leg_direction", "new_small_mode",
    "prior_large_mode", "tmv", "t", "r", "future_outcome",
    "outcome_information_time", "resolution_bars",
]].tail(10)

Clean future-labelled attempts: 1,654


,small_event_id,start_time,completed_leg_direction,new_small_mode,prior_large_mode,tmv,t,r,future_outcome,outcome_information_time,resolution_bars
1644,S2617,2026-10-26 03:15:00+02:00,up,down,up,2.172050,144,0.000038,rejected,2026-10-26 07:35:00+02:00,52
1645,S2619,2026-10-28 20:45:00+02:00,up,down,up,4.246111,765,0.000014,rejected,2026-10-29 12:30:00+02:00,189
1646,S2621,2026-10-29 22:10:00+02:00,up,down,up,1.210491,134,0.000023,propagated,2026-10-30 03:40:00+02:00,66
1647,S2622,2026-10-30 08:20:00+02:00,down,up,down,2.385847,179,0.000033,rejected,2026-10-30 11:15:00+02:00,35
1648,S2624,2026-10-30 18:50:00+02:00,down,up,down,1.773189,76,0.000058,propagated,2026-11-02 02:10:00+02:00,88
1649,S2625,2026-11-04 02:20:00+02:00,up,down,up,4.484562,661,0.000017,propagated,2026-11-04 03:35:00+02:00,15
1650,S2626,2026-11-04 07:00:00+02:00,down,up,down,3.220299,103,0.000078,propagated,2026-11-04 11:05:00+02:00,49
1651,S2627,2026-11-04 19:30:00+02:00,up,down,up,2.992411,130,0.000058,propagated,2026-11-05 06:10:00+02:00,128
1652,S2628,2026-11-05 10:35:00+02:00,down,up,down,2.059971,155,0.000033,rejected,2026-11-05 17:00:00+02:00,77
1653,S2630,2026-11-06 02:25:00+02:00,down,up,down,3.018563,121,0.000062,propagated,2026-11-06 08:45:00+02:00,76


In [13]:
relationship_outcome_audit = small_dcc_relationships.groupby(
    ["is_counter_scale_attempt", "future_outcome", "outcome_is_hindsight_at_dcc"],
    dropna=False,
    sort=False,
).agg(small_dccs=("event_id", "size"))
relationship_outcome_audit

small_dccs
is_counter_scale_attempt future_outcome           outcome_is_hindsight_at_dcc            
False                    NaN                      <NA>                                896
True                     rejected                 True                                894
                         propagated               True                                760
                                                  False                                80
                         unresolved_at_sample_end False                                 1

### Information provenance

The following table is the contract for every later comparison. The future columns may group observations diagnostically but may not be used as inputs at the initiating DCC.

In [14]:
information_contract = pd.DataFrame([
    ("tmv", True, "threshold-normalized movement of the completed aligned small leg"),
    ("t", True, "bars between the completed aligned small leg's extremes"),
    ("r", True, "completed aligned-leg velocity: TMV × theta / T"),
    ("completed_leg_direction", True, "direction that just ended; equals the prior large mode"),
    ("new_small_mode", True, "new counter-scale mode confirmed by the initiating small DCC"),
    ("large_mode_previous_bar", True, "large mode immediately before processing the small-DCC bar"),
    ("large_mode_at_dcc_close", True, "large mode after all same-bar confirmations"),
    ("large_*_previous_bar", True, "live large TMV/T/RDC known on the preceding bar"),
    ("large_*_at_dcc_close", True, "live large TMV/T/RDC after processing the DCC close"),
    ("cross_threshold_relationships", True, "agreement status derived from causal modes"),
    ("future_outcome", False, "propagated or rejected; known only at resolution"),
    ("resolution_bars", False, "future bars required to reach the resolution"),
], columns=["column", "known_at_start", "economic_meaning"])
information_contract

,column,known_at_start,economic_meaning
0,tmv,True,threshold-normalized movement of the completed...
1,t,True,bars between the completed aligned small leg's...
2,r,True,completed aligned-leg velocity: TMV × theta / T
3,completed_leg_direction,True,direction that just ended; equals the prior la...
4,new_small_mode,True,new counter-scale mode confirmed by the initia...
5,large_mode_previous_bar,True,large mode immediately before processing the s...
6,large_mode_at_dcc_close,True,large mode after all same-bar confirmations
7,large_*_previous_bar,True,live large TMV/T/RDC known on the preceding bar
8,large_*_at_dcc_close,True,live large TMV/T/RDC after processing the DCC ...
9,cross_threshold_relationships,True,agreement status derived from causal modes


## Cross-threshold Audit 3: propagation duration and small-event sequence depth

A small DCC strictly between an attempt's start and resolution is impossible under the episode definition: it would itself resolve the attempt as rejection. We assert that count is zero. The meaningful hierarchy statistic is the number of small DCCs—and therefore rejected attempts—between consecutive large DCCs before terminal propagation. That count is almost always odd, but nesting parity is a near-invariant rather than a law: under multiplicative thresholds the large clock can, rarely, confirm a reversal while the small clock still points the other way. Such intervals are flagged and kept out of the rejection-count summary instead of stopping the notebook. Every interval result below becomes known only at the ending large DCC.

In [15]:
small_dcc_positions = small_events["dcc_pos"].to_numpy(dtype=np.int64)
attempt_start_positions = attempts["start_position"].to_numpy(dtype=np.int64)
attempt_resolution_positions = attempts["resolution_position"].to_numpy(dtype=np.int64)
attempts["intervening_small_dccs"] = (
    np.searchsorted(small_dcc_positions, attempt_resolution_positions, side="left")
    - np.searchsorted(small_dcc_positions, attempt_start_positions, side="right")
)
if not attempts["intervening_small_dccs"].eq(0).all():
    raise AssertionError("An attempt contains an intervening small DCC before its resolution.")
attempt_intervening_audit = attempts["intervening_small_dccs"].value_counts().rename_axis(
    "intervening_small_dccs"
).to_frame("attempts")
attempt_intervening_audit

,attempts
intervening_small_dccs,
0,1654


In [16]:
large_intervals = large_events[["event_id", "direction", "dcc_idx", "dcc_pos"]].copy()
large_intervals = large_intervals.rename(columns={
    "event_id": "large_event_id",
    "direction": "large_mode_completed",
    "dcc_idx": "information_time",
})
large_intervals["interval_start_pos"] = large_intervals["dcc_pos"].shift(1)
large_intervals["interval_start_time"] = large_intervals["information_time"].shift(1)
large_intervals = large_intervals.dropna(subset=["interval_start_pos"]).copy()
large_intervals["interval_start_pos"] = large_intervals["interval_start_pos"].astype(np.int64)
left_edges = np.searchsorted(small_dcc_positions, large_intervals["interval_start_pos"], side="right")
right_edges = np.searchsorted(small_dcc_positions, large_intervals["dcc_pos"], side="right")
large_intervals["small_dcc_count"] = right_edges - left_edges
large_intervals["parity_violation"] = (
    large_intervals["small_dcc_count"].lt(1) | large_intervals["small_dcc_count"].mod(2).eq(0)
)
large_intervals["rejected_attempts_before_propagation"] = (
    ((large_intervals["small_dcc_count"] - 1) // 2)
    .astype("Int64")
    .mask(large_intervals["parity_violation"])
)
large_intervals["large_interval_bars"] = (
    large_intervals["dcc_pos"] - large_intervals["interval_start_pos"]
)

propagated_episodes = episodes.loc[episodes["outcome"].eq(AttemptOutcome.PROPAGATED.value), [
    "small_event_id", "start_time", "start_position", "resolution_position", "resolution_bars",
]].rename(columns={
    "small_event_id": "terminal_attempt_event_id",
    "start_time": "terminal_attempt_start_time",
    "start_position": "terminal_attempt_start_pos",
    "resolution_bars": "terminal_propagation_bars",
})
large_intervals = large_intervals.merge(
    propagated_episodes,
    left_on="dcc_pos",
    right_on="resolution_position",
    how="left",
    validate="one_to_one",
)
large_intervals["has_terminal_attempt"] = large_intervals["terminal_attempt_event_id"].notna()
print(
    f"Large intervals: {len(large_intervals):,} | "
    f"parity violations: {int(large_intervals['parity_violation'].sum()):,} | "
    f"without a terminal propagated small attempt: {int((~large_intervals['has_terminal_attempt']).sum()):,}"
)
consistent_intervals = large_intervals.loc[
    ~large_intervals["parity_violation"] & large_intervals["has_terminal_attempt"]
]

large_interval_summary = consistent_intervals.groupby("large_mode_completed", sort=False).agg(
    intervals=("large_event_id", "size"),
    small_dcc_count_median=("small_dcc_count", "median"),
    small_dcc_count_p90=("small_dcc_count", lambda values: values.quantile(0.90)),
    prior_rejections_median=("rejected_attempts_before_propagation", "median"),
    prior_rejections_p90=("rejected_attempts_before_propagation", lambda values: values.quantile(0.90)),
    propagation_bars_median=("terminal_propagation_bars", "median"),
    large_interval_bars_median=("large_interval_bars", "median"),
)
large_interval_summary.style.format(precision=1)

Large intervals: 839 | parity violations: 0 | without a terminal propagated small attempt: 0


,intervals,small_dcc_count_median,small_dcc_count_p90,prior_rejections_median,prior_rejections_p90,propagation_bars_median,large_interval_bars_median
large_mode_completed,,,,,,,
down,420,3.0,7.0,1.0,3,23.0,170.5
up,419,3.0,7.0,1.0,3,20.0,187.0


In [17]:
large_intervals.sort_values(
    ["rejected_attempts_before_propagation", "large_interval_bars"], ascending=False
)[[
    "large_event_id", "large_mode_completed", "interval_start_time", "information_time",
    "small_dcc_count", "rejected_attempts_before_propagation", "terminal_attempt_event_id",
    "terminal_attempt_start_time", "terminal_propagation_bars", "large_interval_bars",
]].head(12)

,large_event_id,large_mode_completed,interval_start_time,information_time,small_dcc_count,rejected_attempts_before_propagation,terminal_attempt_event_id,terminal_attempt_start_time,terminal_propagation_bars,large_interval_bars
749,L751,up,2026-05-06 21:00:00+02:00,2026-06-12 00:30:00+02:00,17,8,S2409,2026-06-11 22:30:00+02:00,24,7530
51,L53,up,2023-08-07 10:00:00+02:00,2023-08-17 15:05:00+02:00,17,8,S209,2023-08-17 13:20:00+02:00,21,2365
152,L154,down,2024-03-04 02:30:00+02:00,2024-03-12 05:15:00+02:00,15,7,S556,2024-03-12 05:05:00+02:00,2,1761
643,L645,up,2025-10-02 20:00:00+02:00,2025-10-10 19:55:00+02:00,15,7,S2037,2025-10-10 08:15:00+02:00,140,1727
653,L655,up,2025-10-21 07:00:00+02:00,2025-10-28 05:30:00+02:00,15,7,S2071,2025-10-28 02:55:00+02:00,31,1422
292,L294,down,2024-09-03 10:15:00+02:00,2024-09-05 19:20:00+02:00,15,7,S958,2024-09-05 19:00:00+02:00,4,685
699,L701,up,2026-02-02 18:15:00+02:00,2026-02-09 04:05:00+02:00,13,6,S2229,2026-02-09 03:45:00+02:00,4,1270
21,L23,up,2023-06-27 11:30:00+02:00,2023-06-30 20:30:00+02:00,13,6,S97,2023-06-30 15:35:00+02:00,59,954
116,L118,down,2023-12-19 15:00:00+02:00,2023-12-22 19:05:00+02:00,13,6,S420,2023-12-22 17:40:00+02:00,17,913
84,L86,down,2023-11-16 15:55:00+02:00,2023-11-21 19:15:00+02:00,13,6,S314,2023-11-21 17:55:00+02:00,16,904


## Outcome Audit 1: robust propagated/rejected summaries

The rows are path-dependent and not IID. We report robust location and quantiles without naive event-level p-values. Any difference remains in-sample diagnostic evidence.

In [18]:
feature_long = attempts.melt(
    id_vars=["future_outcome", "new_small_mode"],
    value_vars=FEATURE_COLUMNS,
    var_name="feature",
    value_name="value",
)
outcome_feature_summary = feature_long.groupby(["future_outcome", "feature"], sort=False)["value"].agg(
    count="size",
    mean="mean",
    p25=lambda values: values.quantile(0.25),
    median="median",
    p75=lambda values: values.quantile(0.75),
    p90=lambda values: values.quantile(0.90),
)
outcome_feature_summary.style.format(precision=6)

,,count,mean,p25,median,p75,p90
future_outcome,feature,,,,,,
rejected,tmv,894,2.892265,1.947455,2.539667,3.511517,4.851659
propagated,tmv,760,2.733989,1.650959,2.439830,3.337242,4.619848
rejected,t,894,133.731544,35.000000,85.000000,169.000000,293.700000
propagated,t,760,108.125000,26.750000,63.000000,136.250000,248.200000
rejected,r,894,0.000162,0.000043,0.000078,0.000160,0.000334
propagated,r,760,0.000230,0.000049,0.000092,0.000219,0.000426


In [19]:
direction_feature_summary = attempts.groupby(["new_small_mode", "future_outcome"], sort=False).agg(
    attempts=("small_event_id", "size"),
    tmv_median=("tmv", "median"),
    t_median=("t", "median"),
    r_median=("r", "median"),
    resolution_bars_median=("resolution_bars", "median"),
)
direction_feature_summary.style.format({
    "attempts": "{:,}",
    "tmv_median": "{:.4f}",
    "t_median": "{:.1f}",
    "r_median": "{:.3e}",
    "resolution_bars_median": "{:.1f}",
})

## Outcome Audit 2: distribution overlap

The logarithms make the right tails visible. Heavy overlap means the completed prior leg does not uniquely determine what the new counter-scale attempt will do.

In [20]:
distribution_fig = plot_dc_attempt_feature_distributions(attempts)
distribution_fig.show()

## Outcome Audit 3: TMV–T geometry and the deterministic role of R

At fixed $\theta_s$, every point satisfies $R=\mathrm{TMV}\theta_s/T$. The dotted contours therefore do not add a third independent dimension; they show how displacement and completion time combine into velocity. Color is the later outcome, while triangle direction is the newly confirmed small mode.

In [21]:
tmv_t_fig = plot_dc_attempt_tmv_t_space(attempts, theta=SMALL_THETA)
tmv_t_fig.show()

## Outcome Audit 4: single-feature overlap scores

Raw AUC is used only as a scale-free overlap description: 0.5 means no monotonic ordering, while values closer to 0 or 1 mean stronger ordering. These are full-sample diagnostics with no fitted model, no OOS claim, and no IID significance test.

In [22]:
is_propagated = attempts["future_outcome"].eq(AttemptOutcome.PROPAGATED.value).astype(np.int8)
overlap_records = []
for feature in FEATURE_COLUMNS:
    raw_auc = roc_auc_score(is_propagated, attempts[feature])
    overlap_records.append({
        "feature": feature,
        "raw_auc_for_higher_value": raw_auc,
        "stronger_value_direction": "higher" if raw_auc >= 0.5 else "lower",
        "orientation_free_auc": max(raw_auc, 1.0 - raw_auc),
    })
overlap_summary = pd.DataFrame(overlap_records).set_index("feature")
overlap_summary.style.format({
    "raw_auc_for_higher_value": "{:.4f}",
    "orientation_free_auc": "{:.4f}",
})

,raw_auc_for_higher_value,stronger_value_direction,orientation_free_auc
feature,,,
tmv,0.4590,lower,0.5410
t,0.4354,lower,0.5646
r,0.5498,higher,0.5498


### Full-sample quintile diagnostics

The cut points below use the entire sample and are therefore not causal thresholds. They answer only whether the realized propagation share changes monotonically across the observed feature range.

In [23]:
quintile_tables = []
for feature in FEATURE_COLUMNS:
    diagnostic_bin = pd.qcut(attempts[feature].rank(method="first"), q=5, labels=False) + 1
    feature_quintiles = attempts.assign(diagnostic_quintile=diagnostic_bin).groupby(
        "diagnostic_quintile", sort=True
    ).agg(
        attempts=("small_event_id", "size"),
        feature_min=(feature, "min"),
        feature_median=(feature, "median"),
        feature_max=(feature, "max"),
        propagation_rate=("future_outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    ).reset_index()
    feature_quintiles.insert(0, "feature", feature)
    quintile_tables.append(feature_quintiles)
quintile_summary = pd.concat(quintile_tables, ignore_index=True)
quintile_summary.style.format({
    "attempts": "{:,}",
    "feature_min": "{:.4g}",
    "feature_median": "{:.4g}",
    "feature_max": "{:.4g}",
    "propagation_rate": "{:.2%}",
})

,feature,diagnostic_quintile,attempts,feature_min,feature_median,feature_max,propagation_rate
0,tmv,1,331,1,1.316,1.611,55.29%
1,tmv,2,331,1.612,2.008,2.224,41.99%
2,tmv,3,330,2.224,2.498,2.796,46.06%
3,tmv,4,331,2.799,3.177,3.771,45.32%
4,tmv,5,331,3.777,4.725,12.02,41.09%
5,t,1,331,1,14,25,54.38%
6,t,2,331,25,38,54,47.43%
7,t,3,330,54,74,102,47.88%
8,t,4,331,102,136,183,41.99%
9,t,5,331,184,272,1877,38.07%


## Outcome Audit 5: chronological stability

A full-sample difference that appears in only one quarter is not a durable structural relationship. Quarterly summaries are still descriptive, but they reveal sign reversals, sparse periods, and concentration.

In [24]:
attempts["calendar_quarter"] = (
    attempts["start_time"].dt.tz_localize(None).dt.to_period("Q").astype(str)
)
quarter_summary = attempts.groupby("calendar_quarter", sort=True).agg(
    attempts=("small_event_id", "size"),
    propagation_rate=("future_outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    tmv_median=("tmv", "median"),
    t_median=("t", "median"),
    r_median=("r", "median"),
)
quarter_summary.style.format({
    "attempts": "{:,}",
    "propagation_rate": "{:.2%}",
    "tmv_median": "{:.4f}",
    "t_median": "{:.1f}",
    "r_median": "{:.3e}",
})

,attempts,propagation_rate,tmv_median,t_median,r_median
calendar_quarter,,,,,
2023Q2,58,37.93%,2.2385,74.0,6.955e-05
2023Q3,101,44.55%,2.5907,107.0,6.252e-05
2023Q4,111,43.24%,2.3395,68.0,8.409e-05
2024Q1,91,40.66%,2.3983,105.0,5.286e-05
2024Q2,66,40.91%,2.4982,126.5,5.714e-05
2024Q3,242,52.07%,2.6682,36.5,1.670e-04
2024Q4,140,47.86%,2.5651,73.5,8.250e-05
2025Q1,165,49.70%,2.3732,62.0,8.677e-05
2025Q2,211,42.65%,2.4987,45.0,1.351e-04


## Outcome Audit 6: nearest cross-outcome pair in causal feature space

To make overlap concrete, we find a propagated and rejected attempt with nearly identical log-TMV and log-T geometry. R is omitted from the distance because it is already determined by those two values. Pair selection uses both hindsight labels and the full sample, so it is a diagnostic casebook—not an evaluation procedure.

In [25]:
geometry = np.log(attempts[["tmv", "t"]].to_numpy(dtype=np.float64))
geometry = (geometry - geometry.mean(axis=0)) / geometry.std(axis=0, ddof=0)
propagated_positions = np.flatnonzero(is_propagated.to_numpy() == 1)
rejected_positions = np.flatnonzero(is_propagated.to_numpy() == 0)
pairwise_squared_distance = (
    (geometry[propagated_positions, None, :] - geometry[rejected_positions, :]) ** 2
).sum(axis=2)
propagated_pair_position, rejected_pair_position = np.unravel_index(
    np.argmin(pairwise_squared_distance), pairwise_squared_distance.shape
)
matched_positions = [
    propagated_positions[propagated_pair_position],
    rejected_positions[rejected_pair_position],
]
matched_pair = attempts.iloc[matched_positions].copy()
matched_pair["standardized_log_geometry_distance"] = np.sqrt(pairwise_squared_distance.min())
matched_pair[[
    "small_event_id", "start_time", "completed_leg_direction", "new_small_mode",
    "tmv", "t", "r", "future_outcome", "resolution_bars",
    "standardized_log_geometry_distance",
]]

,small_event_id,start_time,completed_leg_direction,new_small_mode,tmv,t,r,future_outcome,resolution_bars,standardized_log_geometry_distance
62,S106,2023-07-06 15:15:00+02:00,down,up,1.312704,28,0.000117,propagated,6,0.000782
1332,S2092,2025-11-07 18:20:00+02:00,down,up,1.313193,28,0.000117,rejected,12,0.000782


### Price casebook for the matched pair

The black dotted line is the initiating small DCC, when the completed-leg features become known. The purple dotted line is the later resolution. Read the price path from the black line without looking ahead to the purple line first.

In [26]:
CASEBOOK_BARS_BEFORE = 120
CASEBOOK_BARS_AFTER = 120
for attempt in matched_pair.itertuples(index=False):
    window_start_position = max(0, int(attempt.start_position) - CASEBOOK_BARS_BEFORE)
    window_end_position = min(len(prices), int(attempt.resolution_position) + CASEBOOK_BARS_AFTER + 1)
    case_prices = prices.iloc[window_start_position:window_end_position]
    case_fig = plot_two_threshold_directional_change(
        prices=case_prices,
        small_events=small_events,
        large_events=large_events,
        small_theta=SMALL_THETA,
        large_theta=LARGE_THETA,
        n_prices=len(case_prices),
        title=(
            f"{attempt.future_outcome} | completed {attempt.completed_leg_direction} leg: "
            f"TMV={attempt.tmv:.3f}, T={attempt.t:.0f}, R={attempt.r:.2e}"
        ),
    )
    case_fig.add_vline(x=attempt.start_time, line={"color": "#111827", "dash": "dot", "width": 2})
    case_fig.add_vline(
        x=attempt.resolution_time,
        line={"color": "#7c3aed", "dash": "dot", "width": 2},
    )
    case_fig.show()

## Interpretation checklist

Before moving on, answer these questions from the outputs:

1. Can you reconstruct the large mode and live measurements immediately before and at every small DCC?
2. Can you explain why same-bar large DCCs reset the at-close large measurements?
3. How does small-event TMV/T/R differ between pre-DCC agreement and disagreement?
4. How does live large-scale context differ when the new small mode closes in agreement or disagreement?
5. Why are small-DCC counts between large DCCs almost always odd, how do they encode prior rejected attempts, and when can the large clock confirm first?
6. Are propagated and rejected distributions visibly separated, or mostly overlapping?
7. If R differs, is the difference driven mainly by TMV, T, or both?
8. Is the ordering directionally and chronologically stable?
9. Can nearly identical completed-leg geometry lead to opposite outcomes?
10. Can you state clearly which columns are causal at each DCC and which become known only at resolution?

The next section examines causal **live post-DCC development** of the new counter-scale leg.

# Causal live development of the counter-scale attempt

Completed-event TMV, $T$, and $R$ describe the aligned small leg that ended at the initiating DCC. They barely separate the later outcomes. We now follow the newly confirmed counter-scale leg while it remains unresolved.

This section deliberately repeats the live-leg audit from `2_threshold_dc.ipynb` so the causal-relationship notebook is self-contained. Every live feature is timestamped on the bar when it is knowable. Propagated/rejected colors and groupings remain hindsight diagnostics until the resolution bar.

## Live-attempt semantics and the two clocks

The counter move begins at the peak or trough that the initiating small DCC confirms retrospectively. We therefore retain two causal clocks:

- `small_t_live`: bars from that now-confirmed extreme, including the movement required to confirm the DCC;
- `elapsed_bars`: bars since the initiating DCC, measuring post-confirmation development.

The resolution bar is excluded from the live trajectory. On a rejection, the small live features reset on that bar; on a propagation, the large mode resets. Keeping resolution separate gives both outcomes the same unresolved-state semantics. Same-bar resolutions have no unresolved future life and are excluded.

In [27]:
large_mode_numeric = large_live["direction"]
large_mode_run_id = large_mode_numeric.ne(large_mode_numeric.shift()).cumsum()
large_running_high = prices.groupby(large_mode_run_id).cummax()
large_running_low = prices.groupby(large_mode_run_id).cummin()
large_running_extreme = pd.Series(
    np.where(large_mode_numeric.eq(1.0), large_running_high, large_running_low),
    index=prices.index,
    name="large_running_extreme",
).where(large_mode_numeric.notna())


from dc_basics.two_scale import (
    build_counter_scale_trajectory_panel as _build_counter_scale_trajectory_panel,
)

def build_counter_scale_trajectory_panel(episodes: pd.DataFrame, close: pd.Series,
                                         small_context: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    return _build_counter_scale_trajectory_panel(
        episodes,
        close,
        small_context,
        mode_frame=mode_frame,
        large_running_extreme=large_running_extreme,
        small_theta=SMALL_THETA,
        large_theta=LARGE_THETA,
    )


counter_leg_trajectories, trajectory_attempts = build_counter_scale_trajectory_panel(
    episodes, prices, small_live
)
trajectory_exclusion_audit = pd.Series({
    "all_resolved_attempts": int(episodes["outcome"].isin(
        [AttemptOutcome.PROPAGATED.value, AttemptOutcome.REJECTED.value]
    ).sum()),
    "same_bar_resolutions_excluded": int(episodes["resolution_bars"].eq(0).sum()),
    "trajectory_attempts": len(trajectory_attempts),
    "active_unresolved_bars": len(counter_leg_trajectories),
    "maximum_pre_resolution_large_progress": counter_leg_trajectories["large_dcc_progress"].max(),
    "maximum_elapsed_bars": counter_leg_trajectories["elapsed_bars"].max(),
})
trajectory_exclusion_audit


all_resolved_attempts                     1734.000000
same_bar_resolutions_excluded               80.000000
trajectory_attempts                       1654.000000
active_unresolved_bars                   86630.000000
maximum_pre_resolution_large_progress        0.999888
maximum_elapsed_bars                       735.000000
dtype: float64

## Live information contract

All trajectory features below are causal on their row timestamp. The later outcome may group and color rows diagnostically, but it cannot be used as an input.

In [28]:
live_information_contract = pd.DataFrame([
    ("small_tmv_live", True, "current counter move from the confirmed small extreme"),
    ("small_t_live", True, "bars since that extreme, including pre-DCC confirmation time"),
    ("small_rdc_live", True, "causal running small TMV × theta / T"),
    ("elapsed_bars", True, "bars since the initiating small DCC"),
    ("post_dcc_excursion_theta", True, "signed movement from the DCC close in small-theta units"),
    ("max_post_dcc_excursion_theta", True, "maximum favorable post-DCC excursion observed so far"),
    ("large_dcc_progress", True, "reversal from the causal large running extreme / large theta"),
    ("future_outcome", False, "propagated/rejected; available only at resolution"),
    ("outcome_information_time", False, "timestamp at which the outcome becomes known"),
], columns=["column", "known_on_trajectory_bar", "economic_meaning"])
live_information_contract


,column,known_on_trajectory_bar,economic_meaning
0,small_tmv_live,True,current counter move from the confirmed small ...
1,small_t_live,True,"bars since that extreme, including pre-DCC con..."
2,small_rdc_live,True,causal running small TMV × theta / T
3,elapsed_bars,True,bars since the initiating small DCC
4,post_dcc_excursion_theta,True,signed movement from the DCC close in small-th...
5,max_post_dcc_excursion_theta,True,maximum favorable post-DCC excursion observed ...
6,large_dcc_progress,True,reversal from the causal large running extreme...
7,future_outcome,False,propagated/rejected; available only at resolution
8,outcome_information_time,False,timestamp at which the outcome becomes known


## Live Audit 1: fixed post-DCC checkpoints

At checkpoint $k$, only attempts unresolved after $k$ bars remain. The population changes with $k$: later rows describe increasingly selected survivors, not the original cohort. Outcome-conditioned values are hindsight comparisons among those survivors.

In [29]:
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160]
checkpoint_rows = counter_leg_trajectories.loc[
    counter_leg_trajectories["elapsed_bars"].isin(CHECKPOINT_BARS)
].copy()
checkpoint_survival = checkpoint_rows.groupby("elapsed_bars", sort=True).agg(
    surviving_attempts=("attempt_id", "nunique"),
    later_propagation_rate=("future_outcome", lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean()),
    small_tmv_median=("small_tmv_live", "median"),
    small_t_median=("small_t_live", "median"),
    small_rdc_median=("small_rdc_live", "median"),
    max_post_dcc_excursion_median=("max_post_dcc_excursion_theta", "median"),
    large_dcc_progress_median=("large_dcc_progress", "median"),
)
checkpoint_survival["survivor_share"] = checkpoint_survival["surviving_attempts"] / len(trajectory_attempts)
checkpoint_survival.style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "small_tmv_median": "{:.3f}",
    "small_t_median": "{:.1f}",
    "small_rdc_median": "{:.3e}",
    "max_post_dcc_excursion_median": "{:.3f}",
    "large_dcc_progress_median": "{:.2%}",
    "survivor_share": "{:.2%}",
})


,surviving_attempts,later_propagation_rate,small_tmv_median,small_t_median,small_rdc_median,max_post_dcc_excursion_median,large_dcc_progress_median,survivor_share
elapsed_bars,,,,,,,,
0,"1,654",45.95%,1.076,19.0,1.464e-04,-0.000,54.65%,100.00%
5,"1,487",43.78%,1.063,25.0,9.434e-05,0.107,55.09%,89.90%
10,"1,298",42.68%,1.066,32.0,7.342e-05,0.168,55.07%,78.48%
20,"1,026",42.20%,1.062,45.0,5.261e-05,0.241,55.37%,62.03%
40,672,43.75%,1.053,68.0,3.530e-05,0.314,55.35%,40.63%
80,335,42.99%,1.075,113.0,2.252e-05,0.404,56.11%,20.25%
160,92,45.65%,1.183,208.5,1.292e-05,0.551,62.85%,5.56%


In [30]:
checkpoint_outcome_summary = checkpoint_rows.groupby(
    ["elapsed_bars", "future_outcome", "new_small_mode"], sort=True
).agg(
    surviving_attempts=("attempt_id", "size"),
    small_tmv_median=("small_tmv_live", "median"),
    small_t_median=("small_t_live", "median"),
    small_rdc_median=("small_rdc_live", "median"),
    post_dcc_excursion_median=("post_dcc_excursion_theta", "median"),
    max_post_dcc_excursion_median=("max_post_dcc_excursion_theta", "median"),
    large_dcc_progress_median=("large_dcc_progress", "median"),
)
checkpoint_outcome_summary.style.format(precision=4)


## Live Audit 2: diagnostic overlap at each checkpoint

Orientation-free AUC measures how strongly each live feature orders the two later outcomes among attempts surviving to that checkpoint. It is a hindsight overlap diagnostic, not a causal probability model or an out-of-sample score. Increasing AUC at later checkpoints may reflect both developing structure and survival selection.

In [31]:
LIVE_FEATURE_COLUMNS = [
    "small_tmv_live",
    "small_t_live",
    "small_rdc_live",
    "post_dcc_excursion_theta",
    "max_post_dcc_excursion_theta",
    "large_dcc_progress",
]
live_overlap_records = []
for elapsed_bars, checkpoint in checkpoint_rows.groupby("elapsed_bars", sort=True):
    checkpoint_is_propagated = checkpoint["future_outcome"].eq(
        AttemptOutcome.PROPAGATED.value
    ).astype(np.int8)
    for feature in LIVE_FEATURE_COLUMNS:
        raw_auc = roc_auc_score(checkpoint_is_propagated, checkpoint[feature])
        live_overlap_records.append({
            "elapsed_bars": elapsed_bars,
            "surviving_attempts": len(checkpoint),
            "feature": feature,
            "raw_auc_for_higher_value": raw_auc,
            "stronger_value_direction": "higher" if raw_auc >= 0.5 else "lower",
            "orientation_free_auc": max(raw_auc, 1.0 - raw_auc),
        })
live_overlap_summary = pd.DataFrame(live_overlap_records)
live_overlap_summary.pivot(
    index="elapsed_bars", columns="feature", values="orientation_free_auc"
).style.format("{:.3f}")


feature,large_dcc_progress,max_post_dcc_excursion_theta,post_dcc_excursion_theta,small_rdc_live,small_t_live,small_tmv_live
elapsed_bars,,,,,,
0,0.566,0.500,0.500,0.523,0.520,0.535
5,0.644,0.596,0.637,0.560,0.512,0.634
10,0.677,0.609,0.671,0.594,0.516,0.666
20,0.692,0.605,0.685,0.610,0.507,0.678
40,0.706,0.624,0.699,0.661,0.529,0.694
80,0.670,0.605,0.678,0.649,0.526,0.666
160,0.753,0.608,0.724,0.738,0.594,0.729


## Live Audit 3: survival-conditioned trajectories

Purple/slate denotes the later propagated/rejected outcome; solid/dashed denotes new-small-up/down. Every plotted point was observable at its elapsed bar, but the outcome color remains hindsight. Hover the survivor counts before interpreting later horizons.

In [32]:
trajectory_fig = plot_dc_counter_scale_trajectories(
    counter_leg_trajectories,
    max_elapsed_bars=200,
)
trajectory_fig.show()


## Live Audit 4: price-path casebook

The colored segment contains only unresolved bars and could have been drawn sequentially. Its outcome color is hindsight. The triangle is the initiating small DCC; the black cross is the separate resolution bar. Read from the triangle forward without looking at the cross first.

In [33]:
live_leg_casebook = (
    trajectory_attempts.sort_values("start_time")
    .groupby("future_outcome", sort=False, group_keys=False)
    .tail(1)
    .sort_values("start_time")
)
live_leg_casebook[[
    "attempt_id", "start_time", "new_small_mode", "prior_large_mode",
    "future_outcome", "resolution_time", "resolution_bars",
]]


,attempt_id,start_time,new_small_mode,prior_large_mode,future_outcome,resolution_time,resolution_bars
1732,S2628,2026-11-05 10:35:00+02:00,up,down,rejected,2026-11-05 17:00:00+02:00,77
1733,S2630,2026-11-06 02:25:00+02:00,up,down,propagated,2026-11-06 08:45:00+02:00,76


In [34]:
live_leg_casebook_for_plot = live_leg_casebook
live_leg_casebook_fig = plot_dc_counter_scale_price_casebook(
    prices=prices,
    trajectories=counter_leg_trajectories,
    attempts=live_leg_casebook_for_plot,
    bars_before=120,
    bars_after=120,
)
live_leg_casebook_fig.show()


# Two-threshold causal-relationship completion gate

The phase is complete only when the following distinctions are intuitive:

1. The initiating small DCC makes the **completed prior leg** and the **new opposing mode** knowable at the same timestamp.
2. Completed prior-leg TMV/$T$/$R$ are causal but do not uniquely determine propagation or rejection.
3. The new counter-leg can be measured causally after confirmation using its live geometry and progress toward the large DCC.
4. Propagated/rejected is a future label on every unresolved trajectory row and becomes factual only at resolution.
5. Later checkpoints contain survival-selected cohorts; their outcome rates and AUCs are not initiation-time forecasts.
6. A rejection is confirmed when the small clock reverses first; propagation is confirmed when the large clock changes first.
7. Direct observable structure must be understood before asking whether an HMM or another classifier adds incremental information.

The purpose of these audits is structural understanding, not a trading rule. The next hierarchy milestone is three thresholds, after reproducing the two-threshold conclusions on another market and threshold pairs in `2_threshold_dc_robustness.ipynb`.